In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Define the years to loop through
years = ["2013","2014","2015","2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023"]

def fetch_points_allowed(year):
    url = f"https://www.fantasypros.com/nfl/points-allowed.php?year={year}"
    response = requests.get(url)
    all_teams_data = []
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'html.parser')
        tbody = soup.find('tbody')
        if tbody:
            rows = tbody.find_all('tr')
            for row in rows:
                cells = row.find_all('td')
                if len(cells) >= 12:  # Make sure there are enough cells for all the data
                    all_teams_data.append({
                        'year': year,
                        'team': cells[0].text.strip(),
                        'QB_rank': cells[1].text.strip(),
                        'QB_points': cells[2].text.strip(),
                        'RB_rank': cells[3].text.strip(),
                        'RB_points': cells[4].text.strip(),
                        'WR_rank': cells[5].text.strip(),
                        'WR_points': cells[6].text.strip(),
                        'TE_rank': cells[7].text.strip(),
                        'TE_points': cells[8].text.strip(),
                        'K_rank': None if int(year) < 2015 else cells[9].text.strip(),
                        'K_points': None if int(year) < 2015 else cells[10].text.strip(),
                        'DST_rank': None if int(year) < 2015 else cells[11].text.strip(),
                        'DST_points': None if int(year) < 2015 else cells[12].text.strip(),
                    })
    else:
        print(f"Failed to retrieve data for year {year}")
    return all_teams_data

# Loop through the specified years and collect data
all_data = []
for year in years:
    year_data = fetch_points_allowed(year)
    all_data.extend(year_data)

# Create a DataFrame from the collected data
FPA = pd.DataFrame(all_data)

# Remove any accidental duplicates (sanity check)
FPA = FPA.drop_duplicates()

# Print the DataFrame
print(FPA)
FPA.to_pickle("FPA13-23.pkl")